In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv('heart.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [4]:
df.isna().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [5]:
df['target'].value_counts()

,count
target,
1,165
0,138


In [8]:
X = df.drop(columns='target',axis=1)
y = df['target']

In [9]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numColumns=X.select_dtypes(exclude="object").columns
catColumns=X.select_dtypes(include="object").columns

In [10]:
num_pipline = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())
        ]
    )
cat_pipline = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num",num_pipline,numColumns),
        ("cat",cat_pipline,catColumns)
    ]
)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=3)
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [13]:
X_train.shape, X_test.shape

((242, 13), (61, 13))

In [14]:
y_train.shape, y_test.shape

((242,), (61,))

In [15]:
# list of models
models = [LogisticRegression(max_iter=1000), SVC(kernel='linear'), KNeighborsClassifier(), RandomForestClassifier()]

In [22]:
def compare_models_train_test():
  for model in models:
    model.fit(X_train,y_train)

    y_predict = model.predict(X_test)

    accurecy = accuracy_score(y_test,y_predict)

    print(f"Accurecy for {model} is {accurecy}")

In [23]:
compare_models_train_test()

Accurecy for LogisticRegression(max_iter=1000) is 0.7868852459016393
Accurecy for SVC(kernel='linear') is 0.7704918032786885
Accurecy for KNeighborsClassifier() is 0.8032786885245902
Accurecy for RandomForestClassifier() is 0.7704918032786885


#**Cross Validation**

**LogisticRegression Model**

In [24]:
cross_val_lr = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)

print(cross_val_lr)

cross_val_lr_accurecy = round(np.mean(cross_val_lr)*100,2)

print(f"Accurecy for LogisticRegression is {cross_val_lr_accurecy}%")

[0.80327869 0.86885246 0.85245902 0.86666667 0.75      ]
Accurecy for LogisticRegression is 82.83%


In [25]:
def compare_models_cross_validation():
  for model in models:
    cross_val = cross_val_score(model, X, y, cv=5)

    cross_val_accurecy = round(np.mean(cross_val)*100,2)

    print("Accurecy for ",model,"is",cross_val_accurecy,"%")
    print("------------------------------------------------------")

In [26]:
compare_models_cross_validation()

Accurecy for  LogisticRegression(max_iter=1000) is 82.83 %
------------------------------------------------------
Accurecy for  SVC(kernel='linear') is 82.83 %
------------------------------------------------------
Accurecy for  KNeighborsClassifier() is 64.39 %
------------------------------------------------------
Accurecy for  RandomForestClassifier() is 81.83 %
------------------------------------------------------
